# §38.2: **K5fno** = K5f with normals only from Nexar's negative videos, s2024

Rule: `core/docs/final/KIM_RUN.md` **§38 / §38.1 / §38.2**, written before any K5fno number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. `nexar_test` is never touched.

**Question:** is K5's low TAD micro caused by its normal windows being cut from the accident videos? K5fno keeps K5f's 1,689 positive windows, drops **every** normal window of a positive video, and draws its normals from the 449 negative `nexar_train` videos, each capped at `c` evenly spaced windows (`c` = the cap whose total is closest to the 5,622 in-video normals it replaces). **Config diff to K5f = 0**; only the corpus differs. S0 (source-shortcut probe) passed: S1 0.672 / 0.694 < 0.90. **K5f stays the reference whatever the sign (N6).**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5f run + reads, bakes, census) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 2-3 min |
| 5 | build `Nexar_window_negonly` + its asserts | no | 5-10 min |
| 6 | train **K5fno** (as long as K5f) + N3 checks | yes | 1 × K5f |
| 7 | zero-shot reads: DoTA-CAP-dev (protocol B), DADA whole, TAD, DAD | yes | 20-30 min |
| 8 | `nexar_val` read with `--save-scores` | yes | 10-20 min |
| 9 | K5fno vs K5f: `zs_arm_table` + Nexar val + TAD by category -> **paste `REPORTS/k5fno_report.md` to Claude** | no | 1-2 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim_negonly/` and `Thesis-Final/data/Nexar_window_negonly/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 and its reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '### 38.2 Amendment' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §38.2 -- re-upload kat-vad/'
assert 'def build_window_negonly' in (REPO / 'core/tools/nexar_build.py').read_text(), 'nexar_build has no window_negonly -- re-upload kat-vad/'
assert (REPO / 'core/tools/tad_category_read.py').is_file(), 'no tad_category_read -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/ (training now would reproduce K5)'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # sub-folder name of every eval read (eval/<set>/K5)
K5F      = constants.KIM_FIXED_ARM                 # the control (§31 reference)
K5FNO    = constants.KIM_NEGONLY_ROW               # §38.2
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
C_ONLY   = constants.NEXAR_CONSTRUCTION_WINDOW_NEGONLY
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'     # K5's corpus: the reference of every assert
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics, every train video
# K5f: the control, its run and its reads (§31)
FIXED    = constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME
K5F_RUN  = FIXED / K5F / K5 / f's{SEED}'
K5F_EV   = FIXED / K5F / 'eval'
# Zero-shot inputs: the bakes K5 was read on
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
# Outputs: one folder per row, each read like K5 (eval/<set>/K5, eval/nexar/<row>)
ROOT     = constants.OUTPUT_ROOT / constants.KIM_NEGONLY_OUTPUT_DIRNAME
DATA     = Path(DRIVE_F) / 'data' / f'{DATASET}_{C_ONLY}'         # written in step 5
BUILD    = ROOT / 'REPORTS' / f'build_report_{C_ONLY}.json'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5fno')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


INPUTS = (('Nexar_window labels', WINDOW / constants.LABELS_TRAIN_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5f checkpoint', K5F_RUN / 'checkpoint_last.pt'), ('K5f verified', K5F_RUN / 'verified.json'),
          ('K5f config', K5F_RUN / 'config.yaml'), ('K5f nexar_val read', K5F_EV / 'nexar' / K5F / 'video_stats.json'),
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U),
          ('K5f DoTA read', K5F_EV / 'dota' / K5), ('K5f DADA read', K5F_EV / 'dada' / K5),
          ('K5f TAD read', K5F_EV / 'tad' / K5 / f's{SEED}' / 'scores'),
          ('K5f DAD read', K5F_EV / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | corpus {DATA} (built in step 5) | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-8 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())



# Preflight on small files only (C39)
k5f_cfg = _flat(yaml.safe_load((K5F_RUN / 'config.yaml').read_text()))
assert k5f_cfg['kim.enabled'] and k5f_cfg['kim.prototypes'] and k5f_cfg['kim.edge_mask'], 'K5F_RUN is not K5f'
assert k5f_cfg['dvs.theta'] == 1.0 and k5f_cfg['data.dataset'] == DATASET, 'K5F_RUN is not K5f on Nexar'
STEPS = json.loads((K5F_RUN / 'verified.json').read_text())['step']            # 1,749
EPOCHS = k5f_cfg['train.num_epochs']                                           # 33
assert is_verified(K5F_RUN, STEPS) and metrics_last(K5F_RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5f: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
NEG_IDS = [v for v in TRAIN_IDS if census[v]['label'] == 0]
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert (len(TRAIN_IDS), len(NEG_IDS)) == (900, 449), (len(TRAIN_IDS), len(NEG_IDS))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('Nexar train bake', A2_NEXAR, TRAIN_IDS), ('DADA bake', A2_DADA, DADA_IDS),
                     ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('TAD bake', A2_TAD, TAD_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5f step {STEPS} ({EPOCHS} epochs) | train {len(TRAIN_IDS)} ({len(NEG_IDS)} negative) | '
      f'DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')


## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**


In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_pipeline.py',
     'core/tests/test_tad_category_read.py', 'core/tests/test_nexar_source_probe.py', 'core/tests/test_dvs_dataset.py'])


## 5. Build `Nexar_window_negonly` (§38.2 N2; header reads of the s1 cache only)

Asserts: positive windows and labels identical to `Nexar_window`; **no** normal window from a positive video; every normal window from a negative train video; normal pool within one cap step of the 5,622 in-video normals; val and `defs.json` byte-identical to `Nexar_window`; step plan = K5f's. Prints `c`, the pool, and the pool's scene × weather beside the positives'.


In [ ]:
if not (DATA / constants.LABELS_TRAIN_FILENAME).exists():
    local = LOCAL / 'corpus'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.nexar_build', 'corpus', '--census', CENSUS, '--clip-s1-dir', NEXAR_X,
         '--out-root', local, '--constructions', C_ONLY, '--report', LOCAL / 'build_report.json'], log=LOCAL / 'build.log')
    publish(local / DATA.name, DATA)
    publish(LOCAL / 'build_report.json', BUILD)
    flush_drive()
build = json.loads(BUILD.read_text())
info, PLAN = build['constructions'][C_ONLY], build['plan'][C_ONLY]
assert (PLAN['steps'], PLAN['epochs']) == (STEPS, EPOCHS), f'step plan {PLAN} != K5f {STEPS} = {EPOCHS} epochs'
only = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
win = json.loads((WINDOW / constants.LABELS_TRAIN_FILENAME).read_text())
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
meta = json.loads((DATA / constants.META_FILENAME).read_text())
pos_win = {w: l for w, l in win.items() if l == 1}
assert {w: l for w, l in only.items() if l == 1} == pos_win, 'positive windows differ from Nexar_window (K5f)'
normals = [w for w, l in only.items() if l == 0]
assert {windows[w]['source'] for w in normals} <= set(NEG_IDS), 'a normal window comes from a positive video'
in_video = len(win) - len(pos_win)
cap = info['negative_cap']
assert info['train_in_video_normals_dropped'] == in_video == 5622, info
per_video = {}
for w in normals:
    per_video[windows[w]['source']] = per_video.get(windows[w]['source'], 0) + 1
assert max(per_video.values()) <= cap and abs(len(normals) - in_video) <= len(per_video), (len(normals), cap)
for name in (constants.FRAME_LABELS_TEST_FILENAME, constants.DEFS_FILENAME):
    assert (DATA / name).read_text() == (WINDOW / name).read_text(), f'{name} differs from Nexar_window'


def strata(ids):
    out = {}
    for w in ids:
        key = f"{meta[w].get('scene')}|{meta[w].get('weather')}"
        out[key] = out.get(key, 0) + 1
    total = sum(out.values())
    return {k: round(v / total, 3) for k, v in sorted(out.items(), key=lambda kv: -kv[1])[:8]}


print(f'Nexar_window_negonly: {len(pos_win)} positive (= K5f) | {len(normals)} normal from {len(per_video)} negative videos, '
      f'cap c = {cap} (replaces {in_video} in-video normals) | plan {PLAN}')
print('positives scene|weather shares:', strata(list(pos_win)))
print('normals   scene|weather shares:', strata(normals))


## 6. Train K5fno (§38.2 N1, N3)

**Watch the log: past 2 s/step, stop and paste it.** Then J10, the config diff against K5f (**must be empty**), step-1 `mil` beside K5f's (printed; the normal items differ, so they need not match), last-epoch `mul_mil` above the §31 floor.


In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}']
ARMS = {K5FNO: (K5_SETS, k5f_cfg, K5F_RUN, [])}  # §38.2 N1: K5f's command, only --data-dir differs
DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def loss_print(rows, name):
    keys = ('mil', 'mul_mil', 'kim_mil', 'dvs_sup')
    m = {k: sum(r.get(k, float('nan')) for r in rows) / len(rows) for k in keys}
    print(f'{name}: ' + ' | '.join(f'{k} {m[k]:.4f}' for k in keys))
    return m


def check_run(arm):
    _, base_cfg, base_run, _ = ARMS[arm]
    run_dir, ckpt, _ = row_paths(arm)
    payload = torch.load(ckpt, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(run_dir / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = config_diff(base_cfg, got)
    assert not diff, f'{arm} must have K5f\'s config exactly (§38.2 N1): {diff}'
    rows = [json.loads(l) for l in (run_dir / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    host = [json.loads(l) for l in (base_run / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    first, h1 = loss_print(rows[:1], f'{arm} step 1'), loss_print(host[:1], 'K5f step 1')
    last = loss_print([r for r in rows if r['epoch'] == rows[-1]['epoch']], f'{arm} last epoch')
    loss_print([r for r in host if r['epoch'] == host[-1]['epoch']], 'K5f last epoch')
    print(f'N3 step 1 (printed): mil {first["mil"]:.6f} vs K5f {h1["mil"]:.6f} (normal items differ, need not match)')
    ok = last['mul_mil'] > constants.KIM_FIXED_MUL_MIL_FLOOR
    print(f'N3 §31 check: last-epoch mul_mil {last["mul_mil"]:.4f} vs floor {constants.KIM_FIXED_MUL_MIL_FLOOR} -> '
          f'{"OK" if ok else "FLAG -- paste to Claude"}')


def train_arm(arm):
    sets, _, _, extra = ARMS[arm]
    run_dir, ckpt, _ = row_paths(arm)
    if is_verified(run_dir, STEPS):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        check_run(arm)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / arm
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in (*BASE, *sets) for x in ('--set', s)],
         '--data-dir', stage(DATA, LOCAL / 'stage' / 'data'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *extra, *resume], log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{arm}: trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")}')
    check_run(arm)


train_arm(K5FNO)

## 7. Zero-shot reads of K5fno: DoTA-CAP-dev (protocol B), DADA (whole), TAD, DAD — K5f's protocols and bakes


In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_evaluate(row, name, dataset, data_dir, bake, ids, out, ckpt):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    read_evaluate(row, 'tad', TAD_NAME, L_TAD, A2_TAD, TAD_IDS, ev / 'tad' / K5 / f's{SEED}', ckpt)
    read_evaluate(row, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), A2_DAD, DAD_IDS,
                  ev / constants.DAD_OUTPUT_SET / K5 / f's{SEED}', ckpt)
    flush_drive()


read_zero_shot(K5FNO)

## 8. `nexar_val` read of K5fno with `--save-scores` (`--final` is never passed)


In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


read_nexar(K5FNO)

## 9. K5fno vs K5f -> **paste `REPORTS/k5fno_report.md` to Claude**

`zs_arm_table` (control **K5f**): macro **and** micro on DoTA-CAP-dev and DADA (per-video min-max), TAD and DAD (raw), Δ = K5fno − K5f paired per item; **N5 primary = TAD micro**. Nexar val crops paired per video. TAD by category (`tad_category_read`): macro / clip AUC / micro, `01_Accident` beside the rest. **Printed only (N6): K5f stays the reference.**


In [ ]:
from core.tools import k5_neg_report as kn       # noqa: E402
from core.tools import k5_whole_report as kw     # noqa: E402

local = LOCAL / 'report'
local.mkdir(parents=True, exist_ok=True)
fno_ev = ROOT / K5FNO / 'eval'
NAME = constants.KIM_NEGONLY_REPORT_NAME
reads = []
for arm, ev in ((K5F, K5F_EV), (K5FNO, fno_ev)):
    for s in ('dota', 'dada', 'tad', constants.DAD_OUTPUT_SET):
        reads += ['--read', s, arm, ev / s / K5]
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', K5F, '--seed', SEED, *reads,
     '--name', NAME, '--out-dir', local])
table = (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text()
run([sys.executable, '-m', 'core.tools.tad_category_read', '--read', K5F, K5F_EV / 'tad' / K5 / f's{SEED}',
     '--read', K5FNO, fno_ev / 'tad' / K5 / f's{SEED}', '--name', NAME, '--out-dir', local])
tad_cat = (local / f'{constants.TAD_CATEGORY_STEM}_{NAME}.md').read_text()

# nexar_set keys its control as 'K5' (k5_whole_report); K5f is passed under that key and relabelled below
nx = kw.nexar_set({K5: K5F_EV / 'nexar' / K5F, K5FNO: fno_ev / 'nexar' / K5FNO}, new=K5FNO)
fa = {row: kn.false_alarms(kn.load_video_stats(d))['share']
      for row, d in ((K5F, K5F_EV / 'nexar' / K5F), (K5FNO, fno_ev / 'nexar' / K5FNO))}
dl = nx['delta']
lines = ['# K5fno vs K5f (KIM_RUN.md §38.2), s2024, n = 1', '',
         f"K5fno = K5f config; corpus `{DATA.name}`: {info['train_items'] - len(pos_win)} normal windows from "
         f"{len(per_video)} negative videos (cap {cap}), none from a positive video. Printed only: K5f stays the reference (N6).", '',
         '## Nexar val (crops macro, paired per video)', '',
         '| row | macro crops | macro whole | micro | AP | clip AUC | FA share (peak > 0.5, negatives) |', '|---|---|---|---|---|---|---|']
for key, c in nx['rows'].items():
    row = K5F if key == K5 else key
    lines.append(f"| {row} | {c['macro_crops']['mean']:.4f} | {c['macro_whole']['mean']:.4f} | {c['micro']:.4f} | "
                 f"{c['ap']:.4f} | {c['clip_auc']:.4f} | {fa[row]:.3f} |")
lines += ['', f"K5fno - K5f: {dl['mean']:+.4f} [{dl['low']:+.4f}, {dl['high']:+.4f}] -> {nx['sentence']}", '',
          '## Zero-shot, four sets (N5 primary: TAD micro)', '', table, '', tad_cat]
(local / 'k5fno_report.md').write_text('\n'.join(lines) + '\n')
publish(local, REPORTS)
print((REPORTS / 'k5fno_report.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5fno_report.md to Claude')
